In [4]:
# Part 2: NumPy & Pandas Data Wrangling
import pandas as pd
import numpy as np

# 1. NumPy Arrays & Vectorized Calculations
df_us = pd.read_csv('data/USvideos.csv')
views_arr = df_us['views'].fillna(0).to_numpy(dtype=float)
likes_arr = df_us['likes'].fillna(0).to_numpy(dtype=float)
dislikes_arr = df_us['dislikes'].fillna(0).to_numpy(dtype=float)
comments_arr = df_us['comment_count'].fillna(0).to_numpy(dtype=float)

# Safe engagement calculations
like_ratio = np.divide(likes_arr, (likes_arr + dislikes_arr), out=np.zeros_like(likes_arr), where=(likes_arr + dislikes_arr)!=0)
comment_rate = np.divide(comments_arr, views_arr, out=np.zeros_like(views_arr), where=views_arr!=0)

# Min-max normalization
views_norm = (views_arr - views_arr.min()) / (views_arr.max() - views_arr.min() + 1e-8)
likes_norm = (likes_arr - likes_arr.min()) / (likes_arr.max() - likes_arr.min() + 1e-8)
engagement_score = (0.7 * likes_norm) + (0.3 * views_norm)

top_10_percent_mask = engagement_score >= np.percentile(engagement_score, 90)
print(f"Top 10% engagement records identified: {top_10_percent_mask.sum()}")

# 2. Pandas Loading, Inspection, and Merging
df_us['trending_date'] = pd.to_datetime(df_us['trending_date'], format='%y.%d.%m', errors='coerce')
# Use .dt.tz_localize(None) to strip any timezone information so it matches trending_date
df_us['publish_time'] = pd.to_datetime(df_us['publish_time'], errors='coerce').dt.tz_localize(None)

categories_df = pd.read_csv('data/categories.csv')
df_merged = pd.merge(df_us, categories_df, left_on='category_id', right_on='category_id', how='left')

# 3. Aggregation & Lifespan
df_merged['days_to_trend'] = (df_merged['trending_date'] - df_merged['publish_time'].dt.normalize()).dt.days
trending_lifespan = df_merged.groupby('video_id')['trending_date'].nunique().reset_index(name='trending_lifespan')
print(trending_lifespan.head())

Top 10% engagement records identified: 1608
      video_id  trending_lifespan
0  --j83Wp5WrQ                  2
1  -0dHuLKRPYS                  6
2  -1B196P9MCk                  1
3  -1lyybGIUpD                  4
4  -4iZVX1xNKj                  3
